# Exercise - Multi step Workflow - STARTER

In this exercise, you’ll build a multi-step workflow using LCEL to solve a more complex task than simply generating a joke. 

**Challenge**

Create an AI Business Advisor that:

1. Accepts an industry as input.
2. Generates a business idea.
3. Analyzes the strengths and weaknesses.
4. Formats the results as a final report.


## 0. Import the necessary libs

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

## 1. Configure your API key and instantiate the Chat Model

To connect to OpenAI, instantiate a `ChatOpenAI` client using either of these approaches:

1. Pass the API key directly:

```python
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0,
    api_key='YOUR_API_KEY_HERE',
)
```

2. Store the key in a `.env` file as `OPENAI_API_KEY`, then load it with `python-dotenv`:

```dotenv
OPENAI_API_KEY=YOUR_API_KEY_HERE
```

```python
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0,
)
```

After `load_dotenv()` loads the environment variable, `ChatOpenAI` reads `OPENAI_API_KEY` automatically. Keep the `.env` file private and do not commit it.

In [2]:
# TODO - Instantiate your chat model
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

## 2. Your first Chain

In the end of each chain, you should parse the output and save the logs

In [3]:
logs = []

In [4]:
parser = StrOutputParser()

In [5]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

Make sure {industry} placeholder is inside your template, so it can be filled when the chain is invoked.

In [6]:
# TODO - Your prompt Template
idea_prompt = PromptTemplate(
    template="Generate a business idea for {industry}"
)

In [7]:
# TODO - Create your idea_chain: idea_prompt -> llm -> parse_and_log_output_chain
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [8]:
# TODO - Test your idea_chain invoking it by passing an industry like "agro" to it
idea_result = idea_chain.invoke({"industry": "agro"})

In [9]:
idea_result["output"]

"**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (temperature, humidity, light, and nutrient levels) in real-time.\n   - Develop a mobile app that allows users t

In [10]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (temperature, humidity, light, and nutrient levels) in real-time.\n   - Develop a mobile app 

## 4. Idea Analysis

Craft a prompt to analyze the generated idea's strengths and weaknesses

In [11]:
# TODO - Your prompt Template
analysis_prompt = PromptTemplate(
    template=(
        "You are an expert in business idea analysis. "
        "Given the following business idea: {idea} "
        "Analyze its strengths and weaknesses and provide your opinion."
    )
)

In [12]:
# TODO - Your chain
analysis_chain = analysis_prompt | llm | parse_and_log_output_chain

In [13]:
# TODO - Test your analysis_chain invoking it by passing idea_result["output"] to it
analysis_result = analysis_chain.invoke(
    {"idea": idea_result["output"]}
)

In [14]:
analysis_result["output"]

'### Strengths\n\n1. **Innovative Solution**: The concept of modular vertical farming units addresses the challenges of urban space limitations while providing a sustainable solution for food production. This innovation can attract tech-savvy consumers and urban dwellers looking for fresh produce.\n\n2. **Sustainability Focus**: The emphasis on renewable energy, water recycling, and organic practices aligns with the growing consumer demand for sustainable and environmentally friendly products. This can enhance brand loyalty and attract eco-conscious customers.\n\n3. **Smart Technology Integration**: The use of IoT and a mobile app for monitoring and controlling farming units adds a layer of convenience and efficiency. This tech-driven approach can appeal to a younger demographic and those interested in smart home technologies.\n\n4. **Community Engagement**: By offering workshops and creating a community platform, the business fosters a sense of belonging and encourages knowledge shari

In [15]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (temperature, humidity, light, and nutrient levels) in real-time.\n   - Develop a mobile app 

## 5. Report Generation

Craft a prompt to structure the information into a business report.

In [16]:
# TODO - Your prompt Template
report_prompt = PromptTemplate(
    template=(
        "Convert the following business idea analysis into a structured report. "
        "Select exactly the three most important strengths and exactly the "
        "three most important weaknesses. Rank each list from most to least "
        "important and do not include any additional items.\n\n"
        "Analysis:\n{idea_analysis}"
    )
)

In [17]:
class AnalysisReport(BaseModel):
    """Strengths and weaknesses of a business idea."""

    strengths: list[str] = Field(description="Business idea strengths")
    weaknesses: list[str] = Field(description="Business idea weaknesses")

In [18]:
def parse_and_log_structured_output(result):
    logs.append(result["raw"])
    if result["parsing_error"] is not None:
        raise result["parsing_error"]
    return result["parsed"]


report_chain = (
    report_prompt
    | llm.with_structured_output(AnalysisReport, include_raw=True)
    | RunnableLambda(parse_and_log_structured_output)
)

In [19]:
# TODO - Test your report_chain invoking it by passing analysis_result["output"] to it
report_result = report_chain.invoke({'idea_analysis': analysis_result["output"]})

In [20]:
report_result

AnalysisReport(strengths=['Innovative Solution: The concept of modular vertical farming units addresses the challenges of urban space limitations while providing a sustainable solution for food production. This innovation can attract tech-savvy consumers and urban dwellers looking for fresh produce.', 'Sustainability Focus: The emphasis on renewable energy, water recycling, and organic practices aligns with the growing consumer demand for sustainable and environmentally friendly products. This can enhance brand loyalty and attract eco-conscious customers.', 'Smart Technology Integration: The use of IoT and a mobile app for monitoring and controlling farming units adds a layer of convenience and efficiency. This tech-driven approach can appeal to a younger demographic and those interested in smart home technologies.', 'Community Engagement: By offering workshops and creating a community platform, the business fosters a sense of belonging and encourages knowledge sharing. This can lead t

## 6. End to End Chain

In [24]:
e2e_chain = (
    idea_chain
    | RunnableLambda(
        lambda result: {"idea": result["output"]}
    )
    | analysis_chain
    | RunnableLambda(
        lambda result: {"idea_analysis": result["output"]}
    )
    | report_chain
)

In [25]:
e2e_chain.get_graph().print_ascii()

              +-------------+            
              | PromptInput |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
               +------------+            
               | ChatOpenAI |            
               +------------+            
                      *                  
                      *                  
                      *                  
       +---------------------------+     
       | Parallel<output,log>Input |     
       +---------------------------+     
               ***         ***           
              *               *          
            **                 ** 

In [26]:
# Change the industry if you want
logs.clear()
e2e_result = e2e_chain.invoke({"industry": "agro"})

In [27]:
e2e_result

AnalysisReport(strengths=['Innovative Solution: Modular vertical farming units address urban space limitations and provide sustainable food production.', 'Market Demand: Growing trend towards local, organic produce among health-conscious consumers.', 'Sustainability Focus: Emphasis on renewable energy and water recycling aligns with global sustainability trends.', 'Community Engagement: Workshops and training foster community involvement and build customer loyalty.', 'Diverse Revenue Streams: Multiple revenue streams provide financial stability and growth opportunities.', 'Technological Integration: Use of IoT and mobile apps enhances user experience and optimizes crop yields.'], weaknesses=['High Initial Investment: Significant upfront capital required for design, manufacturing, and technology integration.', 'Technical Complexity: Advanced technology may challenge non-tech-savvy users, necessitating user-friendliness and support.', 'Market Competition: Increasing competition in the ve

In [28]:
e2e_result.strengths

['Innovative Solution: Modular vertical farming units address urban space limitations and provide sustainable food production.',
 'Market Demand: Growing trend towards local, organic produce among health-conscious consumers.',
 'Sustainability Focus: Emphasis on renewable energy and water recycling aligns with global sustainability trends.',
 'Community Engagement: Workshops and training foster community involvement and build customer loyalty.',
 'Diverse Revenue Streams: Multiple revenue streams provide financial stability and growth opportunities.',
 'Technological Integration: Use of IoT and mobile apps enhances user experience and optimizes crop yields.']

In [29]:
e2e_result.weaknesses

['High Initial Investment: Significant upfront capital required for design, manufacturing, and technology integration.',
 'Technical Complexity: Advanced technology may challenge non-tech-savvy users, necessitating user-friendliness and support.',
 'Market Competition: Increasing competition in the vertical farming industry requires differentiation and a competitive edge.',
 'Regulatory Challenges: Urban farming initiatives may face complex zoning laws and health regulations.',
 'Dependency on Urban Markets: Reliance on urban areas limits market potential and exposes the business to economic shifts.',
 'Consumer Education: Potential customers may lack awareness of vertical farming benefits, necessitating effective marketing and education.']

In [30]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (temperature, humidity, light, and nutrient levels) in real-time.\n   - Develop a mobile app 

## 7. Experiment

Now that you understood how it works, experiment with new things.
- Improve memory
- Explore the Runnables
- Add More Complexity